# Stage 3 — GLTR + fine-tuned Transformers + statistics + SHAP + final report

**Run after Stage 2.** Attach the Stage 1 and Stage 2 outputs as Kaggle Inputs. This stage performs the remaining expensive experiments and generates the final CSVs/ZIP.

### Important
This stage is intentionally isolated because fine-tuning DistilBERT/RoBERTa and computing GLTR are the most time-consuming parts of the original notebook. Each model/seed result is checkpointed so a reconnect can resume.

In [20]:

!pip install -q transformers "datasets<4.0.0" accelerate scikit-learn pandas numpy nltk tqdm matplotlib shap statsmodels

import os, re, json, random, warnings, math, pickle, shutil, glob, gc, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm

import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True); nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk import word_tokenize, sent_tokenize, pos_tag
from nltk.corpus import stopwords

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from statsmodels.stats.contingency_tables import mcnemar

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SEEDS = [42,43,44,45,46]
SEEDS_TRANSFORMER = [42,43,44]
STOPWORDS = set(stopwords.words("english"))
FUNCTION_WORDS = ["the","of","and","to","in","is","that","it"]

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
ARTIFACT_DIR = "/kaggle/working/artifacts"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def find_input_file(filename):
    hits = glob.glob("/kaggle/input/**/"+filename, recursive=True)
    if not hits:
        return None
    return hits[0]

print("Working:", os.listdir("/kaggle/working")[:20])


Device: cuda
Working: ['.virtual_documents', 'artifacts', 'checkpoints', 'm4_stage1.pkl', 'm4-hybrid-stage1']


In [21]:

stage1=load_stage1_artifact()
splits=stage1["splits"]; Xs=stage1["Xs"]; Xcat=stage1["Xcat"]; category_dims=stage1["category_dims"]
test_names=[n for n in splits if n not in ("train","val")]

def load_core_results():
    candidates=[os.path.join(ARTIFACT_DIR,"m4_core_results.pkl")]
    candidates += glob.glob("/kaggle/input/**/m4_core_results.pkl",recursive=True)
    for p in candidates:
        if os.path.exists(p):
            print("Loading Stage-2 core results:",p)
            with open(p,"rb") as f: return pickle.load(f)
    raise FileNotFoundError("m4_core_results.pkl not found. Attach the Stage-2 notebook output as Kaggle Input.")

core=load_core_results()
print("Loaded core result groups:",list(core.keys()))


Loading stage-1 artifact: /kaggle/working/artifacts/m4_stage1.pkl
Loading Stage-2 core results: /kaggle/working/artifacts/m4_core_results.pkl
Loaded core result groups: ['hybrid', 'hybrid_raw', 'baselines', 'baseline_preds', 'gates']


In [22]:

def ckpt_path(name):
    return os.path.join(CHECKPOINT_DIR, name + ".pkl")
def ckpt_exists(name):
    return os.path.exists(ckpt_path(name))
def ckpt_save(name, obj):
    tmp = ckpt_path(name) + ".tmp"
    with open(tmp, "wb") as f: pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, ckpt_path(name))
def ckpt_load(name):
    with open(ckpt_path(name), "rb") as f: return pickle.load(f)

def save_artifact(name, obj):
    path = os.path.join(ARTIFACT_DIR, name + ".pkl")
    tmp = path + ".tmp"
    with open(tmp, "wb") as f: pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)
    print("Saved:", path, f"({os.path.getsize(path)/1e6:.2f} MB)")
    return path

def load_stage1_artifact():
    candidates = [os.path.join(ARTIFACT_DIR,"m4_stage1.pkl")]
    hits = glob.glob("/kaggle/input/**/m4_stage1.pkl", recursive=True)
    candidates += hits
    for p in candidates:
        if os.path.exists(p):
            print("Loading stage-1 artifact:", p)
            with open(p,"rb") as f: return pickle.load(f)
    raise FileNotFoundError("m4_stage1.pkl not found. Attach/enable the Output of Stage 1 as Kaggle Input.")


In [23]:

from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_test(labels,preds_a,preds_b):
    labels,preds_a,preds_b=map(np.asarray,(labels,preds_a,preds_b))
    a_ok,b_ok=preds_a==labels,preds_b==labels
    n01=int(np.sum(a_ok & ~b_ok)); n10=int(np.sum(~a_ok & b_ok))
    r=mcnemar([[0,n01],[n10,0]],exact=True)
    return {"A_correct_B_wrong":n01,"A_wrong_B_correct":n10,
            "p_value":r.pvalue,"significant_0.05":r.pvalue<.05}

def bootstrap_ci_f1(labels,preds,n_boot=3000,seed=42):
    rng=np.random.default_rng(seed); labels,preds=np.asarray(labels),np.asarray(preds); n=len(labels)
    vals=[f1_score(labels[idx],preds[idx],zero_division=0) for idx in
          (rng.integers(0,n,n) for _ in range(n_boot))]
    lo,hi=np.percentile(vals,[2.5,97.5])
    return {"f1_mean":float(np.mean(vals)),"f1_ci_lower":float(lo),"f1_ci_upper":float(hi)}


In [24]:

class AttentionGatedHybridDetector(nn.Module):
    def __init__(self,category_dims,bert_dim=768,cat_hidden=32,ffn_out=64):
        super().__init__()
        self.category_names=list(category_dims.keys())
        self.category_encoders=nn.ModuleDict({n:nn.Sequential(nn.Linear(dim,cat_hidden),nn.ReLU(),nn.Dropout(.2))
                                              for n,dim in category_dims.items()})
        n=len(category_dims)
        self.gate=nn.Linear(cat_hidden*n,n)
        self.style_proj=nn.Sequential(nn.Linear(cat_hidden,ffn_out),nn.ReLU())
        self.classifier=nn.Sequential(nn.Linear(bert_dim+ffn_out,256),nn.ReLU(),nn.Dropout(.3),nn.Linear(256,2))
    def forward(self,bert_emb,category_feats,return_gate=False):
        enc=[self.category_encoders[n](category_feats[n]) for n in self.category_names]
        cat=torch.cat(enc,1); weights=torch.softmax(self.gate(cat),1)
        weighted=(torch.stack(enc,1)*weights.unsqueeze(-1)).sum(1)
        style=self.style_proj(weighted)
        logits=self.classifier(torch.cat([bert_emb,style],1))
        return (logits,weights) if return_gate else logits

class CategoryDataset(Dataset):
    def __init__(self,bert_emb,cat_dict,labels):
        self.bert=torch.tensor(bert_emb,dtype=torch.float32)
        self.cat={k:torch.tensor(v,dtype=torch.float32) for k,v in cat_dict.items()}
        self.labels=torch.tensor(labels,dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self,i): return self.bert[i],{k:v[i] for k,v in self.cat.items()},self.labels[i]

def collate_cat(batch):
    be=torch.stack([x[0] for x in batch]); names=batch[0][1].keys()
    cats={n:torch.stack([x[1][n] for x in batch]) for n in names}
    return be,cats,torch.stack([x[2] for x in batch])

def make_loader(bert,cat,labels,batch_size=32,shuffle=True):
    return DataLoader(CategoryDataset(bert,cat,labels),batch_size=batch_size,shuffle=shuffle,collate_fn=collate_cat)

def full_metrics(y,p):
    tn,fp,fn,tp=confusion_matrix(y,p).ravel()
    return {"accuracy":accuracy_score(y,p),"precision":precision_score(y,p,zero_division=0),
            "recall":recall_score(y,p,zero_division=0),"f1":f1_score(y,p,zero_division=0),
            "fpr":fp/(fp+tn) if fp+tn else 0.}

@torch.no_grad()
def eval_gated(model,loader,threshold=.5):
    model.eval(); labs=[]; probs=[]
    for be,cats,lb in loader:
        logits=model(be.to(DEVICE),{k:v.to(DEVICE) for k,v in cats.items()})
        probs.extend(torch.softmax(logits,1)[:,1].cpu().numpy()); labs.extend(lb.numpy())
    labs=np.asarray(labs); probs=np.asarray(probs); return labs,(probs>=threshold).astype(int),probs

def calibrate(labels,probs):
    best=(.5,-1)
    for t in np.linspace(.01,.99,199):
        f=f1_score(labels,(probs>=t).astype(int),zero_division=0)
        if f>best[1]: best=(t,f)
    return best

def train_gated(train_loader,val_loader,category_dims,seed,epochs=15):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model=AttentionGatedHybridDetector(category_dims).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=.01)
    total=len(train_loader)*epochs
    sched=get_cosine_schedule_with_warmup(opt,int(.1*total),total)
    crit=nn.CrossEntropyLoss(); best_f1=-1; best_state=None; no=0
    for ep in range(1,epochs+1):
        model.train()
        for be,cats,lb in train_loader:
            be=be.to(DEVICE); cats={k:v.to(DEVICE) for k,v in cats.items()}; lb=lb.to(DEVICE)
            opt.zero_grad(); loss=crit(model(be,cats),lb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step(); sched.step()
        vl,vp,_=eval_gated(model,val_loader)
        f=f1_score(vl,vp,zero_division=0)
        if f>best_f1:
            best_f1=f; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; no=0
        else:
            no+=1
            if no>=5: break
    model.load_state_dict(best_state)
    vl,_,vprobs=eval_gated(model,val_loader)
    threshold,_=calibrate(vl,vprobs)
    return model,best_f1,threshold

def eval_sklearn(clf,X,y):
    p=clf.predict(X); return full_metrics(y,p),p


In [25]:

stage1=load_stage1_artifact()
splits=stage1["splits"]; Xs=stage1["Xs"]; Xcat=stage1["Xcat"]; category_dims=stage1["category_dims"]
test_names=[n for n in splits if n not in ("train","val")]

bert_tok=AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_model=AutoModel.from_pretrained("distilbert-base-uncased").to(DEVICE).eval()

def make_memmap_embeddings(name,texts,batch_size=16,max_length=256):
    # One .npy per split; written directly to disk so embeddings never accumulate in RAM.
    path=os.path.join(ARTIFACT_DIR,f"m4_bert_{name}.npy")
    if os.path.exists(path):
        print("Using existing:",path)
        return np.load(path,mmap_mode="r")
    shape=(len(texts),768)
    arr=np.lib.format.open_memmap(path,mode="w+",dtype="float32",shape=shape)
    with torch.no_grad():
        for start in tqdm(range(0,len(texts),batch_size),desc=f"BERT {name}"):
            batch=texts[start:start+batch_size]
            enc=bert_tok(batch,padding=True,truncation=True,max_length=max_length,return_tensors="pt")
            enc={k:v.to(DEVICE) for k,v in enc.items()}
            out=bert_model(**enc).last_hidden_state[:,0,:].cpu().numpy().astype("float32")
            arr[start:start+len(out)]=out
            del enc,out
            if start % (batch_size*20)==0: free_memory()
    arr.flush(); del arr
    return np.load(path,mmap_mode="r")

Xb={n:make_memmap_embeddings(n,splits[n].text.tolist()) for n in splits}
free_memory()
del bert_model,bert_tok
free_memory()


Loading stage-1 artifact: /kaggle/working/artifacts/m4_stage1.pkl


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using existing: /kaggle/working/artifacts/m4_bert_train.npy
Using existing: /kaggle/working/artifacts/m4_bert_val.npy
Using existing: /kaggle/working/artifacts/m4_bert_testA.npy
Using existing: /kaggle/working/artifacts/m4_bert_testB.npy
Using existing: /kaggle/working/artifacts/m4_bert_testC.npy


In [26]:

@torch.no_grad()
def gltr_features(text,tok,model,max_len=256):
    ids=tok(text,return_tensors="pt",truncation=True,max_length=max_len).input_ids.to(DEVICE)
    if ids.shape[1]<2: return {"gltr_mean_rank":0.,"gltr_mean_logprob":0.,"gltr_entropy":0.}
    logits=model(ids).logits[0,:-1,:]; targets=ids[0,1:]; lp=torch.log_softmax(logits,-1)
    tlp=lp.gather(1,targets.unsqueeze(1)).squeeze(1)
    ranks=(lp>tlp.unsqueeze(1)).sum(1).float()
    ent=-(lp.exp()*lp).sum(1)
    return {"gltr_mean_rank":ranks.mean().item(),"gltr_mean_logprob":tlp.mean().item(),"gltr_entropy":ent.mean().item()}

def run_gltr():
    from transformers import GPT2LMHeadModel
    tok=AutoTokenizer.from_pretrained("gpt2"); model=GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE).eval()
    Xg={}
    for n,d in splits.items():
        cache=os.path.join(CHECKPOINT_DIR,f"gltr_{n}.pkl")
        if os.path.exists(cache): Xg[n]=pickle.load(open(cache,"rb")); continue
        rows=[]
        for t in tqdm(d.text.tolist(),desc=f"GLTR {n}"):
            rows.append(gltr_features(t,tok,model))
        Xg[n]=pd.DataFrame(rows)
        pickle.dump(Xg[n],open(cache,"wb"),protocol=pickle.HIGHEST_PROTOCOL)
    results={n:[] for n in test_names}; preds={n:[] for n in test_names}
    for seed in SEEDS:
        clf=LogisticRegression(max_iter=2000,random_state=seed).fit(Xg["train"].values,splits["train"].label.values)
        for n in test_names:
            m,p=eval_sklearn(clf,Xg[n].values,splits[n].label.values)
            m.update(model="GLTR (LR)",seed=seed); results[n].append(m); preds[n].append(p)
    del model,tok; free_memory()
    return results,preds

gltr_results,gltr_preds=run_gltr()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GLTR train:   0%|          | 0/8000 [00:00<?, ?it/s]

GLTR val:   0%|          | 0/2000 [00:00<?, ?it/s]

GLTR testA:   0%|          | 0/1000 [00:00<?, ?it/s]

GLTR testB:   0%|          | 0/1000 [00:00<?, ?it/s]

GLTR testC:   0%|          | 0/960 [00:00<?, ?it/s]

In [27]:

class FineTuneDataset(Dataset):
    def __init__(self,texts,labels,tok,max_len=256):
        self.texts=list(texts); self.labels=np.asarray(labels); self.tok=tok; self.max_len=max_len
    def __len__(self): return len(self.labels)
    def __getitem__(self,i):
        e=self.tok(self.texts[i],truncation=True,max_length=self.max_len,padding="max_length",return_tensors="pt")
        return {"input_ids":e["input_ids"][0],"attention_mask":e["attention_mask"][0],
                "label":torch.tensor(int(self.labels[i]),dtype=torch.long)}

class FineTunedClassifier(nn.Module):
    def __init__(self,name):
        super().__init__(); self.encoder=AutoModel.from_pretrained(name); self.classifier=nn.Linear(self.encoder.config.hidden_size,2)
    def forward(self,input_ids,attention_mask):
        return self.classifier(self.encoder(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state[:,0,:])

def train_ft(name,seed,epochs=8,batch_size=8,max_len=256):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    tok=AutoTokenizer.from_pretrained(name); model=FineTunedClassifier(name).to(DEVICE)
    tr=DataLoader(FineTuneDataset(splits["train"].text,splits["train"].label,tok,max_len),batch_size=batch_size,shuffle=True)
    va=DataLoader(FineTuneDataset(splits["val"].text,splits["val"].label,tok,max_len),batch_size=batch_size*2)
    opt=torch.optim.AdamW(model.parameters(),lr=2e-5,weight_decay=.01)
    total=len(tr)*epochs; sched=get_cosine_schedule_with_warmup(opt,int(.1*total),total); crit=nn.CrossEntropyLoss()
    best=-1; state=None; no=0
    for ep in range(1,epochs+1):
        model.train()
        for b in tqdm(tr,desc=f"{name} seed {seed} ep {ep}/{epochs}",leave=False):
            ii,am,lb=b["input_ids"].to(DEVICE),b["attention_mask"].to(DEVICE),b["label"].to(DEVICE)
            opt.zero_grad(); loss=crit(model(ii,am),lb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step(); sched.step()
        model.eval(); vp=[];vl=[]
        with torch.no_grad():
            for b in va:
                ii,am=b["input_ids"].to(DEVICE),b["attention_mask"].to(DEVICE)
                vp.extend(model(ii,am).argmax(1).cpu().numpy()); vl.extend(b["label"].numpy())
        f=f1_score(vl,vp,zero_division=0)
        if f>best: best=f; state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; no=0
        else:
            no+=1
            if no>=3: break
    model.load_state_dict(state)
    return model,tok,best

@torch.no_grad()
def eval_ft(model,tok,texts,labels,batch_size=16,max_len=256):
    ld=DataLoader(FineTuneDataset(texts,labels,tok,max_len),batch_size=batch_size)
    labs=[];preds=[]
    model.eval()
    for b in ld:
        ii,am=b["input_ids"].to(DEVICE),b["attention_mask"].to(DEVICE)
        preds.extend(model(ii,am).argmax(1).cpu().numpy()); labs.extend(b["label"].numpy())
    return np.asarray(labs),np.asarray(preds)

def run_finetuned():
    out={n:[] for n in test_names}; pred={n:{} for n in test_names}
    for base_name,hf_name in [("DistilBERT (fine-tuned)","distilbert-base-uncased"),("RoBERTa (fine-tuned)","roberta-base")]:
        safe=base_name.split()[0].lower()
        for seed in SEEDS_TRANSFORMER:
            cache=os.path.join(CHECKPOINT_DIR,f"ft_{safe}_{seed}.pkl")
            if os.path.exists(cache):
                r=pickle.load(open(cache,"rb"))
            else:
                model,tok,best=train_ft(hf_name,seed)
                r={"best_val_f1":best,"metrics":{},"preds":{}}
                for n in test_names:
                    y,p=eval_ft(model,tok,splits[n].text,splits[n].label)
                    r["metrics"][n]=dict(full_metrics(y,p),model=base_name,seed=seed)
                    r["preds"][n]=p
                pickle.dump(r,open(cache,"wb"),protocol=pickle.HIGHEST_PROTOCOL)
                del model,tok; free_memory()
            for n in test_names:
                out[n].append(r["metrics"][n]); pred[n].setdefault(base_name,[]).append(r["preds"][n])
    return out,pred

finetuned_results,finetuned_preds=run_finetuned()


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilbert-base-uncased seed 42 ep 1/8:   0%|          | 0/1000 [00:00<?, ?it/s]

distilbert-base-uncased seed 42 ep 2/8:   0%|          | 0/1000 [00:00<?, ?it/s]

distilbert-base-uncased seed 42 ep 3/8:   0%|          | 0/1000 [00:00<?, ?it/s]

...

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta-base seed 44 ep 1/8:   0%|          | 0/1000 [00:00<?, ?it/s]

roberta-base seed 44 ep 2/8:   0%|          | 0/1000 [00:00<?, ?it/s]

roberta-base seed 44 ep 3/8:   0%|          | 0/1000 [00:00<?, ?it/s]

roberta-base seed 44 ep 4/8:   0%|          | 0/1000 [00:00<?, ?it/s]

In [29]:

# Statistical tests, SHAP and final CSVs
sig_rows=[]
for n in test_names:
    labels,hybrid_pred=core["hybrid_raw"][n][0]
    comparisons={
        "Stylometric Only (LR)":core["baseline_preds"][n]["Stylometric Only (LR)"][0],
        "BERT-CLS Only (LR)":core["baseline_preds"][n]["BERT-CLS Only (LR)"][0],
        "DistilBERT (fine-tuned)":finetuned_preds[n]["DistilBERT (fine-tuned)"][0],
        "RoBERTa (fine-tuned)":finetuned_preds[n]["RoBERTa (fine-tuned)"][0],
        "GLTR (LR)":gltr_preds[n][0]}
    for base,p in comparisons.items():
        r=mcnemar_test(labels,hybrid_pred,p); r.update(test_set=n,comparison=f"Attention-Gated Hybrid vs {base}"); sig_rows.append(r)
sig_df=pd.DataFrame(sig_rows)

boot_rows=[]
for n in test_names:
    for i,(labels,preds) in enumerate(core["hybrid_raw"][n]):
        r=bootstrap_ci_f1(labels,preds,seed=SEEDS[i]); r.update(test_set=n,seed=SEEDS[i]); boot_rows.append(r)
boot_df=pd.DataFrame(boot_rows)

import shap
shap_clf=LogisticRegression(max_iter=2000,random_state=42).fit(Xs["train"],splits["train"].label.values)
background=shap.sample(Xs["train"],min(200,len(Xs["train"])),random_state=42)
explainer=shap.LinearExplainer(shap_clf,background)
sv=explainer.shap_values(Xs[test_names[0]])
shap_summary=pd.DataFrame({"feature":stage1["feature_columns"],
                           "mean_abs_shap":np.abs(sv).mean(0),
                           "mean_shap":sv.mean(0),
                           "lr_coef":shap_clf.coef_[0]})
shap_summary["coef_shap_sign_match"]=np.sign(shap_summary.lr_coef)==np.sign(shap_summary.mean_shap)
shap_summary=shap_summary.sort_values("mean_abs_shap",ascending=False).reset_index(drop=True)

outdir="/kaggle/working/results_unified"; os.makedirs(outdir,exist_ok=True)
for n in test_names:
    pd.DataFrame(core["hybrid"][n]).to_csv(f"{outdir}/M4_hybrid_{n}.csv",index=False)
    pd.DataFrame(core["baselines"][n]).to_csv(f"{outdir}/M4_baselines_{n}.csv",index=False)
    pd.DataFrame(finetuned_results[n]).to_csv(f"{outdir}/M4_finetuned_{n}.csv",index=False)
    pd.DataFrame(gltr_results[n]).to_csv(f"{outdir}/M4_gltr_{n}.csv",index=False)
sig_df.to_csv(f"{outdir}/M4_mcnemar.csv",index=False)
boot_df.to_csv(f"{outdir}/M4_bootstrap_ci.csv",index=False)
shap_summary.to_csv(f"{outdir}/M4_shap.csv",index=False)
pd.DataFrame(core["gates"]).to_csv(f"{outdir}/M4_attention_gates.csv",index=False)

rows=[]
for n in test_names:
    h=pd.DataFrame(core["hybrid"][n])
    rows.append({"dataset":"M4","test_set":n,"model":"Attention-Gated Hybrid (Ours)",
                 "acc_mean":h.accuracy.mean(),"acc_std":h.accuracy.std(),
                 "f1_mean":h.f1.mean(),"f1_std":h.f1.std(),
                 "fpr_mean":h.fpr.mean(),"fpr_std":h.fpr.std()})
    for src in [core["baselines"][n],finetuned_results[n],gltr_results[n]]:
        d=pd.DataFrame(src)
        for mn,g in d.groupby("model"):
            rows.append({"dataset":"M4","test_set":n,"model":mn,
                         "acc_mean":g.accuracy.mean(),"acc_std":g.accuracy.std(),
                         "f1_mean":g.f1.mean(),"f1_std":g.f1.std(),
                         "fpr_mean":g.fpr.mean(),"fpr_std":g.fpr.std()})
summary=pd.DataFrame(rows).round(4)
summary.to_csv(f"{outdir}/M4_FINAL_SUMMARY.csv",index=False)
print(summary.to_string(index=False))
print("\nTop SHAP features:")
print(shap_summary.head(10).to_string(index=False))
print("\nFiles:",os.listdir(outdir))
!cd /kaggle/working/results_unified && zip -q -r /kaggle/working/M4_results_unified.zip .
print("Created /kaggle/working/M4_results_unified.zip")


dataset test_set                         model  acc_mean  acc_std  f1_mean  f1_std  fpr_mean  fpr_std
     M4    testA Attention-Gated Hybrid (Ours)    0.8842   0.0360   0.8681  0.0476    0.0040   0.0032
     M4    testA            BERT-CLS Only (LR)    0.8630   0.0000   0.8438  0.0000    0.0140   0.0000
     M4    testA         Stylometric Only (LR)    0.7540   0.0000   0.6737  0.0000    0.0000   0.0000
     M4    testA       DistilBERT (fine-tuned)    0.7833   0.1916   0.6684  0.3645    0.0127   0.0042
     M4    testA          RoBERTa (fine-tuned)    0.8810   0.1432   0.8452  0.2075    0.0487   0.0555
     M4    testA                     GLTR (LR)    0.9870   0.0000   0.9871  0.0000    0.0240   0.0000
     M4    testB Attention-Gated Hybrid (Ours)    0.7628   0.0090   0.7137  0.0188    0.0668   0.0176
     M4    testB            BERT-CLS Only (LR)    0.6980   0.0000   0.6308  0.0000    0.1200   0.0000
     M4    testB         Stylometric Only (LR)    0.6670   0.0000   0.5853  0.0000